<div align="center">

# 🚀 LaunchMesh · Image to 3D
### Powered by TripoSG · Free · No GPU required on your machine

**How to use:**
1. Click **Runtime → Run all** (or press `Ctrl+F9`)
2. Wait for the app to launch (~3–5 minutes first time)
3. Use the link that appears to open the LaunchMesh UI
4. Upload your image and generate!

---
</div>

In [ ]:
#@title Step 1 - Install dependencies (auto-restarts runtime) { display-mode: "form" }
import subprocess

def run(cmd, label=""):
    if label: print("  -> " + label)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0: print("  [!] " + r.stderr[-300:])
    return r

print("Installing... (~5 min first time)")

# Clone TripoSG
run("git clone --depth 1 https://github.com/VAST-AI-Research/TripoSG.git /content/TripoSG 2>/dev/null || true", "Cloning TripoSG")

# PyTorch first - diso needs it at build time
run("pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118", "PyTorch")

# diso needs --no-build-isolation to find torch
run("pip install -q diso --no-build-isolation", "diso")

# jaxtyping - required by TripoSG typing utils
run("pip install -q jaxtyping", "jaxtyping")

# Let TripoSG requirements.txt handle diffusers/transformers versions
run("pip install -q -r /content/TripoSG/requirements.txt", "TripoSG requirements")

# Flask + ngrok for the web UI
run("pip install -q flask flask-cors pyngrok", "flask + ngrok")

# Extra utils
run("pip install -q scikit-image jaxtyping", "extra utils")

print("[OK] Done! Restarting runtime...")
print("[!!] After restart: run Steps 2, 3, 4 only - skip Step 1!")

import os
os.kill(os.getpid(), 9)


In [ ]:
#@title  Step 2 — Download TripoSG model weights { display-mode: "form" }
from huggingface_hub import snapshot_download
import os

MODEL_DIR = '/content/checkpoints/TripoSG'
RMBG_DIR  = '/content/checkpoints/RMBG-1.4'

if not os.path.exists(MODEL_DIR):
    print('Downloading TripoSG weights (~2 GB)...')
    snapshot_download('VAST-AI/TripoSG', local_dir=MODEL_DIR)
    print('[OK] TripoSG downloaded!')
else:
    print('[OK] TripoSG already downloaded.')

if not os.path.exists(RMBG_DIR):
    print('Downloading background removal model...')
    snapshot_download('briaai/RMBG-1.4', local_dir=RMBG_DIR)
    print('[OK] RMBG downloaded!')
else:
    print('[OK] RMBG already downloaded.')

print('\n All models ready!')


In [ ]:
#@title  Step 3 — Load models into memory { display-mode: "form" }
import sys, os, torch
sys.path.insert(0, '/content/TripoSG')
sys.path.insert(0, '/content/TripoSG/scripts')

from PIL import Image
import numpy as np
from briarmbg import BriaRMBG
from image_process import prepare_image
from triposg.pipelines.pipeline_triposg import TripoSGPipeline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16
print(f'Using device: {DEVICE}')

print('Loading background removal model...')
rmbg_net = BriaRMBG.from_pretrained('/content/checkpoints/RMBG-1.4').to(DEVICE)
rmbg_net.eval()

print('Loading TripoSG pipeline...')
pipe = TripoSGPipeline.from_pretrained('/content/checkpoints/TripoSG').to(DEVICE, DTYPE)

print('[OK] Models loaded and ready!')


In [ ]:
#@title Step 4 - Launch LaunchMesh App { display-mode: "form" }
NGROK_TOKEN = ""  #@param {type:"string"}

import threading, time, os, uuid
from flask import Flask, request, jsonify, send_file, Response
from flask_cors import CORS
from pyngrok import ngrok
import trimesh, torch
from PIL import Image

flask_app = Flask(__name__)
CORS(flask_app)
os.makedirs("/content/outputs", exist_ok=True)
jobs = {}

@flask_app.route("/")
def index():
    return Response("<h1 style='font-family:sans-serif;padding:40px'>LaunchMesh is running!</h1>", mimetype="text/html")

@flask_app.route("/ping")
def ping():
    return "pong"

# Run Flask - blocking, no thread
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(5000)
print("[OK] URL:", tunnel.public_url)
print("Starting Flask now...")

flask_app.run(host="0.0.0.0", port=5000, use_reloader=False, debug=False)

---
## 💡 Tips for best results
- **Clean background** — white or solid colour behind the object
- **Single object** — one item centred in frame
- **Good lighting** — evenly lit, minimal shadows
- **Square crop** — crop tight around the object

## 📁 Output
Downloads as a `.glb` file — open in **Blender** (File → Import → glTF 2.0), Windows 3D Viewer, Unity, or Unreal.

## ⚡ Free GPU limits
Each person who opens their own copy of this notebook gets their own independent Colab GPU session — no shared quota!

## 🔗 Your shareable link
```
https://colab.research.google.com/github/chopsitv/launchmesh-triposg/blob/main/LaunchMesh_TripoSG.ipynb
```